<a href="https://colab.research.google.com/github/esohman/SoftBCE_Emotions/blob/main/TomatsAgreement%5Bpretty%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tomats Agreement – BERT (hard vs disagreement-soft) + SVM baseline

This notebook is for the downstream classification experiments in a clean order, using the Excel annotation structure:
- one sheet per annotator
- columns: `Text, Emotion1, Intensity, Emotion2, Intensity, Emotion3, Intensity, Other, Note`

It trains/evaluates:
1. **BERT multilabel (hard labels)**
2. **BERT multilabel (soft labels from annotator distributions; cross-entropy / soft-BCE)**
3. **One-vs-rest linear SVM on frozen BERT embeddings**
4. Additionally with **intensity weighted soft labels**.
5. Plots, figures, and graphs

Each is run **with** and **without** Annotator2.


In [ ]:
# !pip -q install transformers torch scikit-learn pandas openpyxl matplotlib

In [ ]:
import numpy as np
import pandas as pd
from typing import Dict, List, Set, Tuple
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier

from transformers import AutoTokenizer, AutoModel

RNG_SEED = 42
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)


## 0. Configuration

In [ ]:
EXCEL_PATH = "TOMATSannotations2025.xlsx"

ANNOTATOR_SHEETS = ["Annotator1-J","Annotator2-N","Annotator3-E"]

TEXT_COL = "Text"

EMOTIONS: List[str] = [
    "anger", "anticipation", "disgust", "fear",
    "joy", "sadness", "surprise", "trust",
]

MODEL_NAME = "bert-base-uncased"  # swap if needed
BATCH_SIZE = 16
LR = 2e-5
MAX_EPOCHS = 10
PATIENCE = 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)


## 1. Load annotations (one sheet per annotator)

In [ ]:
xls = pd.ExcelFile(EXCEL_PATH)

ann_dfs: Dict[str, pd.DataFrame] = {}
for sheet in ANNOTATOR_SHEETS:
    ann_id = sheet.split('-')[0] # Extract "Annotator1" from "Annotator1-J"
    df = pd.read_excel(xls, sheet_name=sheet)
    df[TEXT_COL] = df[TEXT_COL].astype(str)
    ann_dfs[ann_id] = df

print({k: v.shape for k,v in ann_dfs.items()})


In [ ]:
def extract_labelsets_from_sheet(df, emotions):
    """
    df: DataFrame for ONE annotator sheet
    emotions: list of valid Plutchik emotions

    Returns:
      list[set[str]] length N
    """
    valid = set(emotions)
    labelsets = []

    for _, row in df.iterrows():
        labs = set()
        for col in ["Emotion1", "Emotion2", "Emotion3"]:
            val = str(row[col]).strip().lower()
            if val in valid:
                labs.add(val)
        labelsets.append(labs)

    return labelsets


## 2. Parse per-row emotion label sets (handles `none` and Annotator2's `Emotion1` placeholder)

In [ ]:
def clean_token(x) -> str:
    return str(x).strip().lower() if pd.notna(x) else ""

def extract_row_emotions(row: pd.Series, emotion_cols: Tuple[str,str,str]) -> Set[str]:
    e1c,e2c,e3c = emotion_cols
    raw1, raw2, raw3 = clean_token(row.get(e1c,"")), clean_token(row.get(e2c,"")), clean_token(row.get(e3c,""))
    # normalize placeholders
    def norm(tok: str) -> str:
        tok = tok.strip().lower()
        if tok in {"", "none", "nan", "null"}:
            return ""
        return tok
    raw1 = norm(raw1)
    raw2 = norm(raw2)
    raw3 = norm(raw3)
    # Annotator2 sometimes used 'Emotion1' in Emotion2/3 to mean 'same as Emotion1'
    if raw2 == "emotion1": raw2 = raw1
    if raw3 == "emotion1": raw3 = raw1
    out = set()
    for tok in (raw1,raw2,raw3):
        if tok in EMOTIONS:
            out.add(tok)
    return out

def detect_emotion_cols(df: pd.DataFrame) -> Tuple[str,str,str]:
    # Prefer exact names
    for cand in [("Emotion1","Emotion2","Emotion3")]:
        if all(c in df.columns for c in cand):
            return cand
    # fallback: any columns that start with 'Emotion'
    emos = [c for c in df.columns if str(c).lower().startswith("emotion")]
    emos = sorted(emos)
    if len(emos) >= 3:
        return (emos[0], emos[1], emos[2])
    raise ValueError(f"Could not find 3 emotion columns. Columns: {df.columns.tolist()}")

ann_labelsets: Dict[str, List[Set[str]]] = {}
for ann_id, df in ann_dfs.items():
    ecols = detect_emotion_cols(df)
    ann_labelsets[ann_id] = [extract_row_emotions(row, ecols) for _, row in df.iterrows()]

# Build shared item list from Annotator1's Text (assumes aligned ordering across sheets)
df_items = ann_dfs[list(ann_dfs.keys())[0]][[TEXT_COL]].copy()
N = len(df_items)
print("N items:", N)
print("Example labelsets A1/A2/A3:")
for i in [0,1,2]:
    print(i, ann_labelsets["Annotator1"][i], ann_labelsets["Annotator2"][i], ann_labelsets["Annotator3"][i])


In [ ]:
labelsets_by_annotator = {
    ann: extract_labelsets_from_sheet(df, EMOTIONS)
    for ann, df in ann_dfs.items()
}

## 3. Build targets (hard multilabel + soft distribution over annotators)

- **Hard**: label is 1 if *any* included annotator assigned the emotion.
- **Soft**: for each emotion, value is fraction of included annotators assigning it (0..1).


In [ ]:
emo2idx = {e:i for i,e in enumerate(EMOTIONS)}

def build_targets(use_annotator2: bool=True) -> Dict[str, np.ndarray]:
    ann_ids = ["Annotator1","Annotator3"] + (["Annotator2"] if use_annotator2 else [])
    A = len(ann_ids)
    y_counts = np.zeros((N, len(EMOTIONS)), dtype=np.float32)
    for a in ann_ids:
        for i, s in enumerate(ann_labelsets[a]):
            for emo in s:
                y_counts[i, emo2idx[emo]] += 1.0
    y_soft = y_counts / float(A)
    y_hard = (y_counts > 0).astype(np.float32)
    return {"y_soft": y_soft, "y_hard": y_hard, "ann_ids": ann_ids}

targets_withA2 = build_targets(use_annotator2=True)
targets_woA2   = build_targets(use_annotator2=False)
print("Included annotators (with A2):", targets_withA2["ann_ids"])
print("Included annotators (w/o A2):", targets_woA2["ann_ids"])


### 3.1 adding disagreements

In [ ]:
import numpy as np

def labelsets_to_soft_and_hard(labelsets_by_annotator, emotions):
    """
    labelsets_by_annotator: dict[str, list[set[str]]], length N per annotator.
      Each set contains 0..3 emotion labels for that item for that annotator.
    emotions: list[str] length K

    Returns:
      y_soft: (N,K) float, mean distribution across annotators who labeled the item
      y_hard: (N,K) int, union-of-labels (or you can keep your existing hard rule)
      mask_labeled: (N,) bool, item has >=1 label from >=1 annotator
      A_counts: (N,) int, number of annotators contributing (non-empty) per item
    """
    ann_names = list(labelsets_by_annotator.keys())
    N = len(labelsets_by_annotator[ann_names[0]])
    K = len(emotions)
    emo2i = {e:i for i,e in enumerate(emotions)}

    y_soft = np.zeros((N, K), dtype=np.float32)
    y_hard = np.zeros((N, K), dtype=np.int32)
    A_counts = np.zeros(N, dtype=np.int32)

    for i in range(N):
        acc = np.zeros(K, dtype=np.float32)
        n_contrib = 0

        for a in ann_names:
            labs = labelsets_by_annotator[a][i]
            if not labs:
                continue
            v = np.zeros(K, dtype=np.float32)
            for e in labs:
                if e in emo2i:
                    v[emo2i[e]] = 1.0
            s = v.sum()
            if s > 0:
                v = v / s  # annotator mass split across their labels
                acc += v
                n_contrib += 1
                y_hard[i] = np.maximum(y_hard[i], (v > 0).astype(np.int32))

        if n_contrib > 0:
            y_soft[i] = acc / n_contrib
            A_counts[i] = n_contrib

    mask_labeled = A_counts > 0
    return y_soft, y_hard, mask_labeled, A_counts


## 4. Train/validation split (shared across conditions)

In [ ]:
idx = np.arange(N)
train_idx, val_idx = train_test_split(idx, test_size=0.2, random_state=RNG_SEED, shuffle=True)
print(len(train_idx), len(val_idx))


## 5. BERT multilabel model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TomatsDataset(Dataset):
    def __init__(self, df: pd.DataFrame, indices: np.ndarray, y_hard: np.ndarray, y_soft: np.ndarray):
        self.texts = df.iloc[indices][TEXT_COL].tolist()
        self.y_hard = torch.tensor(y_hard[indices], dtype=torch.float32)
        self.y_soft = torch.tensor(y_soft[indices], dtype=torch.float32)
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        enc = tokenizer(self.texts[i], truncation=True, padding='max_length', max_length=256, return_tensors='pt')
        item = {k: v.squeeze(0) for k,v in enc.items()}
        item['y_hard'] = self.y_hard[i]
        item['y_soft'] = self.y_soft[i]
        return item

class BertMultiLabel(nn.Module):
    def __init__(self, model_name: str, num_labels: int):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
    def forward(self, input_ids, attention_mask=None, token_type_ids=None):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        cls = out.last_hidden_state[:,0]
        return self.classifier(cls)


### Losses

- **Hard BCE**: `BCEWithLogitsLoss` with 0/1 targets.
- **Soft disagreement loss**: same BCE form but using **soft targets** in [0,1] (cross-entropy / soft-label BCE).


In [ ]:
bce = nn.BCEWithLogitsLoss()

def hard_bce_loss(logits, y_hard):
    return bce(logits, y_hard)

def soft_bce_loss(logits, y_soft):
    # Cross-entropy between Bernoulli targets and sigmoid(logits)
    return bce(logits, y_soft)


### Training + evaluation helpers

In [ ]:
@torch.no_grad()
def collect_probs(model, loader) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    y_true = []
    y_prob = []
    for batch in loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        token_type_ids = batch.get('token_type_ids')
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(DEVICE)
        logits = model(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        prob = torch.sigmoid(logits).cpu().numpy()
        y_prob.append(prob)
        y_true.append(batch['y_hard'].cpu().numpy())
    return np.vstack(y_true), np.vstack(y_prob)

def best_threshold_for_macro_f1(y_true, y_prob, thresholds=None):
    if thresholds is None:
        thresholds = np.linspace(0.1, 0.9, 17)
    best_t, best = 0.5, -1.0
    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
        if f1 > best:
            best = f1
            best_t = float(t)
    return best_t, best

def eval_f1(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    out = {
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'micro_f1': f1_score(y_true, y_pred, average='micro', zero_division=0),
    }
    per = {}
    for k, emo in enumerate(EMOTIONS):
        per[emo] = f1_score(y_true[:,k], y_pred[:,k], average='binary', zero_division=0)
    return out, per, y_pred


In [ ]:
def train_bert(
    targets: Dict[str, np.ndarray],
    loss_kind: str,
    label: str,
    soft_key: str = "y_soft"   # <-- NEW
):

    train_ds = TomatsDataset(df_items, train_idx, targets['y_hard'], targets[soft_key])
    val_ds   = TomatsDataset(df_items, val_idx,   targets['y_hard'], targets[soft_key])

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    model = BertMultiLabel(MODEL_NAME, len(EMOTIONS)).to(DEVICE)
    optim = torch.optim.AdamW(model.parameters(), lr=LR)
    best_state = None
    best_macro = -1
    patience_left = PATIENCE

    for epoch in range(1, MAX_EPOCHS+1):
        model.train()
        total = 0.0
        for batch in train_loader:
            optim.zero_grad()
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            token_type_ids = batch.get('token_type_ids')
            if token_type_ids is not None:
                token_type_ids = token_type_ids.to(DEVICE)
            logits = model(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
            if loss_kind == 'hard':
                loss = hard_bce_loss(logits, batch['y_hard'].to(DEVICE))
            elif loss_kind == 'soft':
                loss = soft_bce_loss(logits, batch['y_soft'].to(DEVICE))
            else:
                raise ValueError(loss_kind)
            loss.backward()
            optim.step()
            total += float(loss.item())
        # val
        y_true, y_prob = collect_probs(model, val_loader)
        t_best, macro_best = best_threshold_for_macro_f1(y_true, y_prob)
        print(f"[{label}] epoch {epoch} loss {total/len(train_loader):.4f} val_macro_f1 {macro_best:.4f} @t={t_best:.2f}")
        if macro_best > best_macro + 1e-4:
            best_macro = macro_best
            best_state = {k: v.cpu().clone() for k,v in model.state_dict().items()}
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left <= 0:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, val_loader


In [ ]:
def _safe_normalize_rows(P, eps=1e-12):
    P = np.asarray(P, dtype=np.float64)
    row_sums = P.sum(axis=1, keepdims=True)
    row_sums = np.maximum(row_sums, eps)
    return P / row_sums

def ce(p, q, eps=1e-12):
    """
    Cross-entropy H(p,q) row-wise for distributions (N,K).
    """
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    q = np.clip(q, eps, 1.0)
    return -(p * np.log(q)).sum(axis=1)

def ce_disagreement_per_item(labelsets_by_annotator, emotions):
    """
    Computes per-item CE disagreement:
      D(i) = average_a H(p_a(i), p_bar(i)) over annotators with non-empty labels.
    Returns:
      D: (N,) float (np.nan where no annotators labeled)
      p_bar: (N,K) mean distribution across contributing annotators
      A_counts: (N,) number of contributing annotators
    """
    ann_names = list(labelsets_by_annotator.keys())
    N = len(labelsets_by_annotator[ann_names[0]])
    K = len(emotions)
    emo2i = {e:i for i,e in enumerate(emotions)}

    # Build annotator distributions p_a(i)
    P_sum = np.zeros((N, K), dtype=np.float64)
    A_counts = np.zeros(N, dtype=np.int32)

    P_anns = {a: np.zeros((N, K), dtype=np.float64) for a in ann_names}
    contrib = {a: np.zeros(N, dtype=bool) for a in ann_names}

    for a in ann_names:
        for i in range(N):
            labs = labelsets_by_annotator[a][i]
            if not labs:
                continue
            v = np.zeros(K, dtype=np.float64)
            for e in labs:
                if e in emo2i:
                    v[emo2i[e]] = 1.0
            s = v.sum()
            if s > 0:
                v = v / s
                P_anns[a][i] = v
                contrib[a][i] = True
                P_sum[i] += v
                A_counts[i] += 1

    p_bar = np.zeros((N, K), dtype=np.float64)
    mask = A_counts > 0
    p_bar[mask] = P_sum[mask] / A_counts[mask, None]
    p_bar = _safe_normalize_rows(p_bar)  # should already sum to 1 for mask rows

    # Now compute avg CE to mean
    D = np.full(N, np.nan, dtype=np.float64)
    if mask.any():
        ce_acc = np.zeros(N, dtype=np.float64)
        for a in ann_names:
            m = contrib[a]
            if m.any():
                ce_acc[m] += ce(P_anns[a][m], p_bar[m])
        D[mask] = ce_acc[mask] / A_counts[mask]

    return D, p_bar.astype(np.float32), A_counts


## 6. Train BERT models (hard vs soft) with and without Annotator2

In [ ]:
y_soft_all, y_hard_all, mask_labeled, A_counts = labelsets_to_soft_and_hard(labelsets_by_annotator, EMOTIONS)


In [ ]:
texts = ann_dfs["Annotator1"]["Text"].tolist()


In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

RNG_SEED = 42
N = len(texts)
idx = np.arange(N)

# 1) Hold out TEST set (never touched during training or threshold tuning)
trainval_idx, test_idx = train_test_split(
    idx,
    test_size=0.2,
    random_state=RNG_SEED,
    shuffle=True
)

# 2) Split remaining into TRAIN and VAL
train_idx, val_idx = train_test_split(
    trainval_idx,
    test_size=0.2,
    random_state=RNG_SEED,
    shuffle=True
)

print(
    "train:", len(train_idx),
    "val:", len(val_idx),
    "test:", len(test_idx)
)


In [ ]:
labelsets_withA2 = labelsets_by_annotator

labelsets_woA2 = {
    k: v for k, v in labelsets_by_annotator.items()
    if k != "Annotator2"
}


In [ ]:
def slice_labelsets(labelsets_by_annotator, idx):
    return {
        a: [labelsets_by_annotator[a][i] for i in idx]
        for a in labelsets_by_annotator
    }

labelsets_withA2_val  = slice_labelsets(labelsets_withA2, val_idx)
labelsets_withA2_test = slice_labelsets(labelsets_withA2, test_idx)

labelsets_woA2_val  = slice_labelsets(labelsets_woA2, val_idx)
labelsets_woA2_test = slice_labelsets(labelsets_woA2, test_idx)


In [ ]:
#sanity check
print(labelsets_withA2_val.keys())
print(labelsets_woA2_val.keys())
print(len(next(iter(labelsets_withA2_val.values()))), len(val_idx))


## 7. Evaluate BERT models

In [ ]:
results = []
per_emotion_tables = {}

for name, (model, vl, split_labelsets) in models.items():
    # existing: hard labels + model probs
    y_true, y_prob = collect_probs(model, vl)

    # 1) thresholds + F1 (existing)
    t_best, _ = best_threshold_for_macro_f1(y_true, y_prob)
    agg, per, y_pred = eval_f1(y_true, y_prob, threshold=t_best)

    # 2) CE disagreement on THIS split
    D_ce, p_bar, A_counts = ce_disagreement_per_item(split_labelsets, EMOTIONS)
    ce_dis_mean = float(np.nanmean(D_ce))
    ce_dis_median = float(np.nanmedian(D_ce))

    # 3) Model-to-annotator soft cross-entropy
    # y_prob is N x K probs
    # For multilabel sigmoid outputs, normalize across emotions for this diagnostic:
    y_prob_norm = _safe_normalize_rows(y_prob)
    ce_model = ce(p_bar, y_prob_norm)
    # Only evaluate where we have soft labels (A_counts>0)
    mask = A_counts > 0
    ce_model_mean = float(np.mean(ce_model[mask])) if mask.any() else np.nan
    ce_model_median = float(np.median(ce_model[mask])) if mask.any() else np.nan

    row = {
        "model": name,
        "threshold": float(t_best),
        **agg,
        "ce_disagree_mean": ce_dis_mean,
        "ce_disagree_median": ce_dis_median,
        "ce_model_mean": ce_model_mean,
        "ce_model_median": ce_model_median,
        "n_soft_items": int(mask.sum()),
        "n_annotators_avg": float(np.mean(A_counts[mask])) if mask.any() else np.nan,
    }
    results.append(row)
    per_emotion_tables[name] = per

pd.DataFrame(results).sort_values("macro_f1", ascending=False)


## 8. SVM baseline on frozen BERT embeddings (one-vs-rest)

We extract CLS embeddings using a frozen BERT encoder and train a linear OVR SVM for multilabel prediction.


In [ ]:
@torch.no_grad()
def cls_embeddings(texts: List[str], batch_size: int = 32) -> np.ndarray:
    enc_model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
    enc_model.eval()
    out_emb = []
    for i in range(0, len(texts), batch_size):
        batch_text = texts[i:i+batch_size]
        enc = tokenizer(batch_text, truncation=True, padding=True, max_length=256, return_tensors='pt')
        enc = {k:v.to(DEVICE) for k,v in enc.items()}
        h = enc_model(**enc).last_hidden_state[:,0].cpu().numpy()
        out_emb.append(h)
    return np.vstack(out_emb)

def train_eval_svm(targets: Dict[str,np.ndarray], label: str):
    X_train = cls_embeddings(df_items.iloc[train_idx][TEXT_COL].tolist())
    X_val   = cls_embeddings(df_items.iloc[val_idx][TEXT_COL].tolist())
    y_train = targets['y_hard'][train_idx]
    y_val   = targets['y_hard'][val_idx]
    clf = OneVsRestClassifier(LinearSVC())
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_val)
    macro = f1_score(y_val, y_pred, average='macro', zero_division=0)
    micro = f1_score(y_val, y_pred, average='micro', zero_division=0)
    print(f"[{label}] macro_f1={macro:.4f} micro_f1={micro:.4f}")
    return {'model': label, 'macro_f1': macro, 'micro_f1': micro}

svm_res = []
svm_res.append(train_eval_svm(targets_withA2, 'SVM OVR WITH A2'))
svm_res.append(train_eval_svm(targets_woA2, 'SVM OVR WITHOUT A2'))
pd.DataFrame(svm_res)


## 9. Single-label confusion matrix (diagnostic)

For a compact confusion matrix, we convert multilabel targets/predictions to a single label by taking the argmax of predicted probabilities and argmax of the (hard) target vector.


In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

def plot_confusion_singlelabel(model, vl, title: str):
    y_true, y_prob = collect_probs(model, vl)
    true_single = np.argmax(y_true, axis=1)
    pred_single = np.argmax(y_prob, axis=1)
    cm = confusion_matrix(true_single, pred_single, labels=range(len(EMOTIONS)))
    cm = cm / np.clip(cm.sum(axis=1, keepdims=True), 1, None)
    plt.figure(figsize=(7,6))
    plt.imshow(cm, aspect='auto')
    plt.xticks(range(len(EMOTIONS)), EMOTIONS, rotation=45, ha='right')
    plt.yticks(range(len(EMOTIONS)), EMOTIONS)
    plt.colorbar(label='Row-normalized')
    plt.title(title)
    plt.tight_layout()
    plt.show()


df_res = pd.DataFrame(results).sort_values('macro_f1', ascending=False)
best_name = df_res.iloc[0]['model']
print('Best:', best_name)
best_model, best_vl, best_labelsets = models[best_name]
plot_confusion_singlelabel(best_model, best_vl, f"Confusion (single-label diagnostic): {best_name}")


## 10. Annotator confusion labels


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def annotation_overlap_matrix(labelsets_a, labelsets_b, emotions):
    K = len(emotions)
    emo2i = {e: i for i, e in enumerate(emotions)}
    M = np.zeros((K, K), dtype=np.int32)

    for la, lb in zip(labelsets_a, labelsets_b):
        for ea in la:
            if ea not in emo2i:
                continue
            for eb in lb:
                if eb not in emo2i:
                    continue
                M[emo2i[ea], emo2i[eb]] += 1

    return M


In [ ]:
def row_normalize(M):
    return M / np.clip(M.sum(axis=1, keepdims=True), 1, None)


In [ ]:
def plot_annotation_overlap(M, emotions, title):
    plt.figure(figsize=(7,6))
    plt.imshow(M, aspect='auto')
    plt.xticks(range(len(emotions)), emotions, rotation=45, ha='right')
    plt.yticks(range(len(emotions)), emotions)
    plt.colorbar(label='Row-normalized overlap')
    plt.title(title)
    plt.tight_layout()
    plt.show()


In [ ]:
M13 = annotation_overlap_matrix(
    labelsets_by_annotator["Annotator1"],
    labelsets_by_annotator["Annotator3"],
    EMOTIONS
)

M13_norm = row_normalize(M13)

plot_annotation_overlap(
    M13_norm,
    EMOTIONS,
    "Annotation overlap (Annotator 1 vs Annotator 3)"
)


## 11. Final addition of intensities as additional soft weights

In [ ]:
import numpy as np
import pandas as pd

def _to_float(x):
    try:
        if pd.isna(x):
            return np.nan
        return float(x)
    except Exception:
        return np.nan

def extract_labelweights_from_sheet_scaled01(df, emotions=EMOTIONS, eps=1e-8):
    """
    Returns list of dicts: per row {emotion: scaled_weight_in_[0,1]}.
    Scaling is done per sheet (annotator) using min-max over all intensity entries.
    """
    valid = set(e.lower() for e in emotions)

    emo_cols = ["Emotion1", "Emotion2", "Emotion3"]

    # Find all intensity columns robustly (handles Intensity, Intensity.1, Intensity.2)
    int_cols = [c for c in df.columns if str(c).lower().startswith("intensity")]
    # Fallback: if someone renamed them oddly, keep the original assumption
    if len(int_cols) < 3:
        int_cols = ["Intensity", "Intensity.1", "Intensity.2"]

    # Collect all numeric intensities to get min/max for this annotator sheet
    all_vals = []
    for c in int_cols:
        if c in df.columns:
            all_vals.extend([_to_float(v) for v in df[c].values])
    all_vals = np.array([v for v in all_vals if np.isfinite(v)], dtype=float)

    if all_vals.size == 0:
        sheet_min, sheet_max = 0.0, 1.0
    else:
        sheet_min = float(all_vals.min())
        sheet_max = float(all_vals.max())

    denom = max(sheet_max - sheet_min, eps)

    weights_per_row = []
    for _, row in df.iterrows():
        wdict = {}

        for ec, ic in zip(emo_cols, int_cols[:3]):
            if ec not in df.columns or ic not in df.columns:
                continue

            e = str(row[ec]).strip().lower()
            if e not in valid:
                continue

            w = _to_float(row[ic])
            if not np.isfinite(w):
                continue

            # min-max scale to [0,1]
            w01 = (float(w) - sheet_min) / denom
            w01 = float(np.clip(w01, 0.0, 1.0))

            if w01 > 0:
                wdict[e] = w01

        weights_per_row.append(wdict)

    return weights_per_row


In [ ]:
def build_targets_from_labelweights(labelweights_by_annotator, emotions, alpha_uniform=0.0, eps=1e-12):
    """
    labelweights_by_annotator: dict[annotator -> list[dict[emotion->weight]]], length N per annotator
    emotions: list[str] length K

    Returns:
      y_soft_int: (N,K) intensity-weighted soft labels
      y_soft_uni: (N,K) uniform soft labels (for comparison)
      y_hard:     (N,K) union-of-labels hard target
      A_counts:   (N,) number of contributing annotators per item
    alpha_uniform:
      0.0 = pure intensity-based
      >0 mixes in uniform mass for stability: p = (1-a)*p_int + a*p_uniform
    """
    ann = list(labelweights_by_annotator.keys())
    N = len(labelweights_by_annotator[ann[0]])
    K = len(emotions)
    emo2i = {e.lower(): i for i, e in enumerate(emotions)}

    y_soft_int = np.zeros((N, K), dtype=np.float32)
    y_soft_uni = np.zeros((N, K), dtype=np.float32)
    y_hard = np.zeros((N, K), dtype=np.int32)
    A_counts = np.zeros(N, dtype=np.int32)

    for i in range(N):
        acc_int = np.zeros(K, dtype=np.float64)
        acc_uni = np.zeros(K, dtype=np.float64)
        n_contrib = 0

        for a in ann:
            wdict = labelweights_by_annotator[a][i]
            if not wdict:
                continue

            # intensity distribution
            v_int = np.zeros(K, dtype=np.float64)
            # uniform distribution over selected emotions
            v_uni = np.zeros(K, dtype=np.float64)

            items = [(e.lower(), w) for e, w in wdict.items() if e.lower() in emo2i and np.isfinite(w)]
            if not items:
                continue

            # fill vectors
            for e, w in items:
                idx = emo2i[e]
                v_int[idx] += max(float(w), 0.0)
                v_uni[idx] = 1.0

            # normalize
            s_int = v_int.sum()
            s_uni = v_uni.sum()
            if s_uni <= 0:
                continue

            v_uni = v_uni / s_uni
            if s_int > 0:
                v_int = v_int / s_int
            else:
                # fallback: if all intensities were zero, revert to uniform
                v_int = v_uni.copy()

            # optional mixing for robustness
            if alpha_uniform > 0:
                v_int = (1 - alpha_uniform) * v_int + alpha_uniform * v_uni

            acc_int += v_int
            acc_uni += v_uni
            n_contrib += 1

            y_hard[i] = np.maximum(y_hard[i], (v_uni > 0).astype(np.int32))

        if n_contrib > 0:
            y_soft_int[i] = (acc_int / n_contrib).astype(np.float32)
            y_soft_uni[i] = (acc_uni / n_contrib).astype(np.float32)
            A_counts[i] = n_contrib

    return y_soft_int, y_soft_uni, y_hard, A_counts


In [ ]:
# WITH A2
labelweights_withA2 = {
    ann: extract_labelweights_from_sheet_scaled01(ann_dfs[ann], EMOTIONS)
    for ann in ["Annotator1", "Annotator2", "Annotator3"]
}

y_soft_int_withA2, y_soft_uni_withA2, y_hard_withA2, A_counts_withA2 = build_targets_from_labelweights(
    labelweights_withA2,
    EMOTIONS,
    alpha_uniform=0.0  # try 0.1 if intensities are noisy
)

# WITHOUT A2
labelweights_woA2 = {
    ann: extract_labelweights_from_sheet_scaled01(ann_dfs[ann])
    for ann in ["Annotator1", "Annotator3"]
}

y_soft_int_woA2, y_soft_uni_woA2, y_hard_woA2, A_counts_woA2 = build_targets_from_labelweights(
    labelweights_woA2, EMOTIONS, alpha_uniform=0.0
)


In [ ]:
def sheet_intensity_summary(df):
    cols = [c for c in df.columns if str(c).lower().startswith("intensity")]
    vals = []
    for c in cols:
        vals.extend([_to_float(v) for v in df[c].values])
    vals = np.array([v for v in vals if np.isfinite(v)], dtype=float)
    return dict(n=int(vals.size), min=float(vals.min()), max=float(vals.max()), mean=float(vals.mean()))

for ann in ann_dfs:
    print(ann, sheet_intensity_summary(ann_dfs[ann]))


In [ ]:
targets_withA2 = {
    "y_hard": y_hard_withA2,
    "y_soft": y_soft_uni_withA2,     # uniform
    "y_soft_int": y_soft_int_withA2  # intensity-weighted
}


In [ ]:
# all models
models = {}

m, vl = train_bert(
    targets_withA2,
    loss_kind="hard",
    label="BERT-hard WITH A2"
)
models["bert_hard_withA2"] = (m, vl, labelsets_withA2_val)

m, vl = train_bert(
    targets_withA2,
    loss_kind="soft",
    label="BERT-soft (uniform) WITH A2",
    soft_key="y_soft"
)
models["bert_soft_uniform_withA2"] = (m, vl, labelsets_withA2_val)

m, vl = train_bert(
    targets_withA2,
    loss_kind="soft",
    label="BERT-soft (intensity) WITH A2",
    soft_key="y_soft_int"
)
models["bert_soft_intensity_withA2"] = (m, vl, labelsets_withA2_val)

m, vl = train_bert(
    targets_woA2,
    loss_kind="hard",
    label="BERT-hard WITHOUT A2"
)
models["bert_hard_woA2"] = (m, vl, labelsets_woA2_val)

m, vl = train_bert(
    targets_woA2,
    loss_kind="soft",
    label="BERT-soft (uniform) WITHOUT A2",
    soft_key="y_soft"
)
models["bert_soft_uniform_woA2"] = (m, vl, labelsets_woA2_val)

m, vl = train_bert(
    targets_woA2,
    loss_kind="soft",
    label="BERT-soft (intensity) WITHOUT A2",
    soft_key="y_soft_int"
)
models["bert_soft_intensity_woA2"] = (m, vl, labelsets_woA2_val)



In [ ]:
from sklearn.metrics import f1_score

macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

def evaluate_all_models(models, emotions):
    results = []
    per_emotion_tables = {}

    for name, pack in models.items():
        if len(pack) == 2:
            model, vl = pack
            labelsets_val = None
        else:
            model, vl, labelsets_val = pack

        # probs + hard ground truth from the loader
        y_true, y_prob = collect_probs(model, vl)

        # choose threshold on THIS split
        t_best, _ = best_threshold_for_macro_f1(y_true, y_prob)

        # hard preds
        y_pred = (y_prob >= t_best).astype(int)

        macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
        micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)

        # per-emotion F1 table
        try:
            per = per_emotion_f1_table(y_true, y_pred, emotions)
        except Exception:
            per = None

        # --- CE diagnostics ---
        ce_dis_mean = ce_dis_median = np.nan
        ce_model_mean = ce_model_median = np.nan
        n_soft_items = 0
        n_annotators_avg = np.nan

        if labelsets_val is not None:
            D_ce, p_bar, A_counts = ce_disagreement_per_item(labelsets_val, emotions)

            # mask items where soft dist defined (>=1 annotator contributed)
            mask = (A_counts > 0)

            if mask.any():
                ce_dis_mean = float(np.mean(D_ce[mask]))
                ce_dis_median = float(np.median(D_ce[mask]))

                ce_model = ce_model_to_annotdist(y_prob, p_bar)  # vector length N
                ce_model_mean = float(np.mean(ce_model[mask]))
                ce_model_median = float(np.median(ce_model[mask]))

                n_soft_items = int(mask.sum())
                n_annotators_avg = float(np.mean(A_counts[mask]))

        row = {
            "model": name,
            "threshold": float(t_best),
            "macro_f1": float(macro_f1),
            "micro_f1": float(micro_f1),
            "ce_disagree_mean": ce_dis_mean,
            "ce_disagree_median": ce_dis_median,
            "ce_model_mean": ce_model_mean,
            "ce_model_median": ce_model_median,
            "n_soft_items": int(n_soft_items),
            "n_annotators_avg": n_annotators_avg,
        }
        results.append(row)
        per_emotion_tables[name] = per

    df_res = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
    return df_res, per_emotion_tables


In [ ]:
import numpy as np

def ce_model_to_annotdist(y_prob: np.ndarray, p_bar: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """
    Per-item cross-entropy between mean annotator distribution p_bar (N,K)
    and model distribution q derived from y_prob (N,K).

    Returns:
        ce: shape (N,)
    """
    # Turn model multilabel probs into a distribution over emotions
    q = y_prob.astype(np.float64)
    q = q / np.clip(q.sum(axis=1, keepdims=True), eps, None)  # row-normalize
    q = np.clip(q, eps, 1.0)  # avoid log(0)

    p = p_bar.astype(np.float64)
    p = p / np.clip(p.sum(axis=1, keepdims=True), eps, None)  # safety (should already sum to 1)

    ce = -np.sum(p * np.log(q), axis=1)
    return ce


In [ ]:
print("models:", list(models.keys()))  # sanity check
df_res, per_emotion_tables = evaluate_all_models(models, EMOTIONS)
df_res


# Checks

In [ ]:
print("Have y_soft_int_withA2?", "y_soft_int_withA2" in globals())
print("Have y_soft_int_woA2?", "y_soft_int_woA2" in globals())
print("Have y_soft_uni_woA2?", "y_soft_uni_woA2" in globals())
print("Have y_hard_woA2?", "y_hard_woA2" in globals())


In [ ]:
targets_woA2 = {
    "y_hard": y_hard_woA2,
    "y_soft": y_soft_uni_woA2,       # uniform
    "y_soft_int": y_soft_int_woA2    # intensity-weighted
}


In [ ]:
print("targets_withA2 keys:", targets_withA2.keys())
print("targets_woA2 keys:", targets_woA2.keys())


# Re-doing soft labels

In [ ]:
import numpy as np
import pandas as pd

XLSX_PATH = "TOMATSannotations2025.xlsx"
SHEETS = {
    "Annotator1": "Annotator1-J",
    "Annotator2": "Annotator2-N",
    "Annotator3": "Annotator3-E",
}
TEXT_COL = "Text"
EMOTIONS = ["anger","anticipation","disgust","fear","joy","sadness","surprise","trust"]

def _clean(tok):
    if pd.isna(tok):
        return ""
    tok = str(tok).strip().lower()
    return "" if tok in {"", "none", "nan", "null"} else tok

def load_annotations(xlsx_path=XLSX_PATH):
    ann_labelsets = {}
    ann_ints = {}
    texts = None

    for ann, sheet in SHEETS.items():
        df = pd.read_excel(xlsx_path, sheet_name=sheet)
        if texts is None:
            texts = df[TEXT_COL].astype(str).tolist()

        labelsets = []
        intensities = []  # list of dict emo->intensity

        for _, row in df.iterrows():
            labs = set()
            ints = {}
            for emo_col, int_col in [("Emotion1","Intensity"),("Emotion2","Intensity.1"),("Emotion3","Intensity.2")]:
                emo = _clean(row.get(emo_col, ""))
                if emo and emo in EMOTIONS:
                    labs.add(emo)
                    v = row.get(int_col, np.nan)
                    if pd.notna(v):
                        ints[emo] = float(v)
            labelsets.append(labs)
            intensities.append(ints)

        ann_labelsets[ann] = labelsets
        ann_ints[ann] = intensities

    return texts, ann_labelsets, ann_ints

def labelsets_to_binary(labelsets):
    emo2i = {e:i for i,e in enumerate(EMOTIONS)}
    Y = np.zeros((len(labelsets), len(EMOTIONS)), dtype=np.int32)
    for i, s in enumerate(labelsets):
        for e in s:
            Y[i, emo2i[e]] = 1
    return Y

def calibrate_intensities_per_annotator(ann_ints):
    """
    Returns per-annotator calibration parameters for intensity normalization.
    Simple approach: min-max per annotator over all observed intensities.
    """
    calib = {}
    for ann, ints_list in ann_ints.items():
        vals = []
        for d in ints_list:
            vals.extend(list(d.values()))
        if len(vals) == 0:
            calib[ann] = (0.0, 10.0)
        else:
            calib[ann] = (float(np.min(vals)), float(np.max(vals)))
    return calib

def make_targets_variant(texts, ann_labelsets, ann_ints,
                         include_ann=("Annotator1","Annotator3","Annotator2"),
                         variant="frac",  # "frac", "masssplit", "intensity_raw", "intensity_calib", "reliability_weighted"
                         reliability_weights=None,
                         intensity_calib=None):
    """
    Returns:
      y_soft: (N,K) float32 in [0,1] (interpretation depends on variant)
      y_hard: (N,K) float32 in {0,1}
    """
    N = len(texts)
    K = len(EMOTIONS)
    emo2i = {e:i for i,e in enumerate(EMOTIONS)}

    include_ann = [a for a in include_ann if a in ann_labelsets]
    A = len(include_ann)

    # hard = union-of-labels across included annotators
    y_hard = np.zeros((N, K), dtype=np.float32)
    for a in include_ann:
        Ya = labelsets_to_binary(ann_labelsets[a])
        y_hard = np.maximum(y_hard, Ya.astype(np.float32))

    if variant == "frac":
        # fraction of annotators selecting each emotion (multilabel Bernoulli probability)
        counts = np.zeros((N, K), dtype=np.float32)
        for a in include_ann:
            counts += labelsets_to_binary(ann_labelsets[a]).astype(np.float32)
        y_soft = counts / float(A)

    elif variant == "masssplit":
        # each annotator splits 1.0 mass across their chosen labels; then mean across annotators who chose >=1
        y_soft = np.zeros((N, K), dtype=np.float32)
        denom = np.zeros(N, dtype=np.float32)
        for a in include_ann:
            for i, labs in enumerate(ann_labelsets[a]):
                if not labs:
                    continue
                v = np.zeros(K, dtype=np.float32)
                for e in labs:
                    v[emo2i[e]] = 1.0
                v = v / max(v.sum(), 1.0)
                y_soft[i] += v
                denom[i] += 1.0
        denom = np.clip(denom, 1.0, None)
        y_soft = (y_soft.T / denom).T

    elif variant in {"intensity_raw", "intensity_calib"}:
        # intensity-weighted multilabel probability per emotion:
        # y_soft[i,k] = mean_a (I(a,i,k)/10) over annotators who selected k
        y_soft = np.zeros((N, K), dtype=np.float32)
        counts = np.zeros((N, K), dtype=np.float32)

        if variant == "intensity_calib":
            if intensity_calib is None:
                intensity_calib = calibrate_intensities_per_annotator(ann_ints)

        for a in include_ann:
            for i, labs in enumerate(ann_labelsets[a]):
                if not labs:
                    continue
                for e in labs:
                    raw = ann_ints[a][i].get(e, np.nan)
                    if np.isnan(raw):
                        continue
                    if variant == "intensity_raw":
                        v = raw / 10.0
                    else:
                        mn, mx = intensity_calib[a]
                        # min-max then clip
                        v = 0.0 if mx <= mn else (raw - mn) / (mx - mn)
                        v = float(np.clip(v, 0.0, 1.0))
                    y_soft[i, emo2i[e]] += v
                    counts[i, emo2i[e]] += 1.0

        # mean intensity among annotators who selected emotion (0 if nobody selected)
        counts = np.clip(counts, 1.0, None)
        y_soft = y_soft / counts

    elif variant == "reliability_weighted":
        # weighted fraction-of-annotators by reliability_weights (implicitly handles missingness)
        if reliability_weights is None:
            raise ValueError("Provide reliability_weights dict, e.g. {'Annotator1':1.0,'Annotator3':1.0,'Annotator2':0.1}")
        wsum = 0.0
        y_soft = np.zeros((N, K), dtype=np.float32)
        for a in include_ann:
            w = float(reliability_weights.get(a, 1.0))
            y_soft += w * labelsets_to_binary(ann_labelsets[a]).astype(np.float32)
            wsum += w
        y_soft = y_soft / max(wsum, 1e-6)

    else:
        raise ValueError(f"Unknown variant: {variant}")

    return y_soft.astype(np.float32), y_hard.astype(np.float32)

def disagreement_score_frac(y_soft_frac):
    """
    Per-item entropy of the soft fractions, normalized by log K (as in your paper).
    NOTE: this treats y_soft as a distribution over K, so it renormalizes rows.
    """
    eps = 1e-12
    row_sums = np.sum(y_soft_frac, axis=1, keepdims=True)
    P = y_soft_frac / np.clip(row_sums, eps, None)
    H = -np.sum(P * np.log(np.clip(P, eps, 1.0)), axis=1)
    return H / np.log(len(EMOTIONS))

def latex_example_table(texts, ann_labelsets, y_soft, y_hard,
                        include_ann=("Annotator1","Annotator3","Annotator2"),
                        n_rows=6, pick="high_disagreement"):
    """
    Produces a Leonardelli-style illustrative table (not a results table).
    """
    include_ann = [a for a in include_ann if a in ann_labelsets]
    Ys = {a: labelsets_to_binary(ann_labelsets[a]) for a in include_ann}

    if pick == "high_disagreement":
        # use disagreement from y_soft as a heuristic
        d = disagreement_score_frac(y_soft)
        idx = np.argsort(-d)[:n_rows]
    else:
        rng = np.random.default_rng(0)
        idx = rng.choice(len(texts), size=n_rows, replace=False)


    def vec_str(v):
        return "[" + ",".join(str(int(x)) for x in v.tolist()) + "]"

    def soft_str(v):
        return "[" + ",".join(f"{x:.2f}" for x in v.tolist()) + "]"

    lines = []
    lines.append(r"\begin{table*}[t]")
    lines.append(r"\centering")
    colspec = "p{0.45\\textwidth}" + "c"*len(include_ann) + "c" + "c"
    lines.append(rf"\begin{{tabular}}{{{colspec}}}")
    lines.append(r"\hline")
    header = ["Example"] + [f"{a}" for a in include_ann] + ["Soft labels"] + ["Hard labels"]
    lines.append(" & ".join(header) + r" \\")
    lines.append(r"\hline")

    for i in idx:
        ex = texts[i].replace("\n"," ")
        if len(ex) > 140:
            ex = ex[:137] + "..."
        row = [ex]
        for a in include_ann:
            row.append(vec_str(Ys[a][i]))
        row.append(soft_str(y_soft[i]))
        row.append(vec_str(y_hard[i]))
        lines.append(" & ".join(row) + r" \\")
    lines.append(r"\hline")
    lines.append(r"\end{tabular}")
    lines.append(r"\caption{Examples illustrating individual multilabel annotations, aggregated soft targets, and aggregated hard targets. Emotions are ordered as: "
                 + ", ".join(EMOTIONS) + r".}")
    lines.append(r"\end{table*}")

    return "\n".join(lines)

# --- RUN ---
texts, ann_labelsets, ann_ints = load_annotations()

# suggested reliability weights that heavily downweight sparse annotator
coverage = {a: np.mean([len(s)>0 for s in ann_labelsets[a]]) for a in ann_labelsets}
reliability_weights = {a: coverage[a] for a in coverage}

# Build several variants
variants = ["frac", "masssplit", "intensity_raw", "intensity_calib", "reliability_weighted"]
targets = {}
for v in variants:
    y_soft, y_hard = make_targets_variant(
        texts, ann_labelsets, ann_ints,
        include_ann=("Annotator1","Annotator3","Annotator2"),
        variant=v,
        reliability_weights=reliability_weights
    )
    targets[v] = (y_soft, y_hard)


# Appendix

In [ ]:
import numpy as np
import pandas as pd
from itertools import combinations

def cooccurrence_matrix(labelsets, emotions, include_diagonal=True, normalize=False):
    """
    labelsets: List[Set[str]]  (one set per item for a single annotator)
    emotions:  List[str]       (fixed order)
    include_diagonal: whether to count single-label occurrences on diagonal
    normalize: if True, convert counts to proportions by total items (N)

    Returns: (matrix (np.ndarray), df (pd.DataFrame))
    """
    idx = {e: i for i, e in enumerate(emotions)}
    M = np.zeros((len(emotions), len(emotions)), dtype=np.int64)

    for s in labelsets:
        labs = [e for e in s if e in idx]
        if not labs:
            continue

        # diagonal = frequency of each label appearing in an item
        if include_diagonal:
            for e in labs:
                M[idx[e], idx[e]] += 1

        # off-diagonal = pair co-occurrence within the same item
        for a, b in combinations(sorted(labs), 2):
            i, j = idx[a], idx[b]
            M[i, j] += 1
            M[j, i] += 1

    if normalize:
        N = len(labelsets)
        M = M.astype(np.float64) / max(N, 1)

    df = pd.DataFrame(M, index=emotions, columns=emotions)
    return M, df


def top_k_pairs(cooc_df, k=10):
    """
    Returns the top-k co-occurring *pairs* (off-diagonal), sorted by count.
    """
    emotions = list(cooc_df.index)
    rows = []
    for i in range(len(emotions)):
        for j in range(i+1, len(emotions)):
            rows.append((emotions[i], emotions[j], int(cooc_df.iat[i, j])))
    out = pd.DataFrame(rows, columns=["emotion_a", "emotion_b", "count"])
    return out.sort_values("count", ascending=False).head(k).reset_index(drop=True)



cooc_dfs = {}
top_pairs = {}

for ann_id, labelsets in ann_labelsets.items():
    _, df_cooc = cooccurrence_matrix(labelsets, EMOTIONS, include_diagonal=True, normalize=False)
    cooc_dfs[ann_id] = df_cooc
    top_pairs[ann_id] = top_k_pairs(df_cooc, k=10)

print("Annotator1 co-occurrence matrix:")
print(cooc_dfs["Annotator1"])

print("\nTop co-occurring pairs (Annotator1):")
print(top_pairs["Annotator1"])

for ann_id, df_cooc in cooc_dfs.items():
    df_cooc.to_csv(f"cooccurrence_{ann_id}.csv", index=True)

## heatmap

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_cooccurrence_heatmap(cooc_df, title="", log_scale=False, show_values=False):
    """
    cooc_df: pd.DataFrame (square, emotions x emotions)
    log_scale: if True, plots log1p(counts) for better contrast
    show_values: if True, prints counts in cells (can get cluttered)
    """
    data = cooc_df.to_numpy()
    plot_data = np.log1p(data) if log_scale else data

    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    im = ax.imshow(plot_data, aspect='equal')  # default colormap

    ax.set_title(title)
    ax.set_xticks(range(cooc_df.shape[1]))
    ax.set_yticks(range(cooc_df.shape[0]))
    ax.set_xticklabels(cooc_df.columns, rotation=45, ha='right')
    ax.set_yticklabels(cooc_df.index)

    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("log1p(count)" if log_scale else "count")

    if show_values:
        for i in range(cooc_df.shape[0]):
            for j in range(cooc_df.shape[1]):
                ax.text(j, i, str(int(data[i, j])), ha='center', va='center', fontsize=7)

    plt.tight_layout()
    plt.show()




for ann_id, df_cooc in cooc_dfs.items():
    plot_cooccurrence_heatmap(
        df_cooc,
        title=f"Emotion Co-occurrence (Diagonal=frequency) — {ann_id}",
        log_scale=True,        # helps when counts are skewed
        show_values=False      # set True if matrices are small / sparse
    )

In [ ]:
df_no_diag = df_cooc.copy()
np.fill_diagonal(df_no_diag.values, 0)
plot_cooccurrence_heatmap(df_no_diag, title=f"Pair Co-occurrence (no diagonal) — {ann_id}", log_scale=True)

# appendix

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_cooccurrence_heatmap(
    cooc_df,
    title="",
    log_scale=False,
    show_values=False,
    cmap="magma",
    vmin=None,
    vmax=None
):
    data = cooc_df.to_numpy()
    plot_data = np.log1p(data) if log_scale else data

    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    im = ax.imshow(
        plot_data,
        aspect="equal",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax
    )

    ax.set_title(title)
    ax.set_xticks(range(cooc_df.shape[1]))
    ax.set_yticks(range(cooc_df.shape[0]))
    ax.set_xticklabels(cooc_df.columns, rotation=45, ha="right")
    ax.set_yticklabels(cooc_df.index)

    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("log1p(count)" if log_scale else "count")

    if show_values:
        for i in range(cooc_df.shape[0]):
            for j in range(cooc_df.shape[1]):
                ax.text(j, i, str(int(data[i, j])), ha="center", va="center", fontsize=7)

    plt.tight_layout()
    plt.show()


# Examples:
plot_cooccurrence_heatmap(cooc_dfs["Annotator1"], title="Annotator1", log_scale=True, cmap="Reds")
plot_cooccurrence_heatmap(cooc_dfs["Annotator1"], title="Annotator1", log_scale=True, cmap="viridis")
plot_cooccurrence_heatmap(cooc_dfs["Annotator1"], title="Annotator1", log_scale=True, cmap="cividis")
plot_cooccurrence_heatmap(cooc_dfs["Annotator1"], title="Annotator1", log_scale=True, cmap="YlOrRd")

In [ ]:
df_no_diag = cooc_dfs["Annotator1"].copy()
np.fill_diagonal(df_no_diag.values, 0)
plot_cooccurrence_heatmap(df_no_diag, title="Annotator1 (no diagonal)", log_scale=True, cmap="Reds")

In [ ]:
vmax = max(np.log1p(df.values).max() for df in cooc_dfs.values())

for ann_id, df in cooc_dfs.items():
    df_plot = df.copy()
    np.fill_diagonal(df_plot.values, 0)
    plot_cooccurrence_heatmap(
        df_plot,
        title=f"{ann_id} (no diagonal)",
        log_scale=True,
        cmap="magma",
        vmin=0,
        vmax=vmax
    )

# extracting other emotions






In [ ]:
import re
import pandas as pd
from collections import Counter
from typing import Dict, List, Tuple, Optional

# --- helpers ---

PLACEHOLDERS = {"", "none", "nan", "null", "n/a", "na", "-", "—"}

def clean_tok(x) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip().lower()

def find_col(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    """Find first matching column name (case-insensitive) among candidates."""
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    return None

def detect_other_note_cols(df: pd.DataFrame) -> Tuple[Optional[str], Optional[str]]:
    other_col = find_col(df, ["Other", "Other emotion", "Other emotions", "OtherEmotion", "Other_Emotion"])
    note_col  = find_col(df, ["Note", "Notes", "Comment", "Comments", "Rationale"])
    return other_col, note_col

def split_other_field(s: str) -> List[str]:
    """
    Split free-text other field into candidate labels.
    Handles comma/semicolon/slash/newline separation and strips junk.
    """
    s = clean_tok(s)
    if s in PLACEHOLDERS:
        return []
    parts = re.split(r"[,;/\n]+", s)
    out = []
    for p in parts:
        p = p.strip().lower()
        p = re.sub(r"\s+", " ", p)
        if p and p not in PLACEHOLDERS:
            out.append(p)
    return out

def extract_non_plutchik_from_emotion_cols(df: pd.DataFrame, emotions: List[str]) -> Counter:
    """
    Collect non-Plutchik tokens appearing in Emotion1/2/3 (excluding placeholders).
    Also normalizes the special case where annotator wrote 'Emotion1' in Emotion2/3.
    """
    valid = set([e.lower() for e in emotions])
    ctr = Counter()

    # robustly locate emotion columns (matches your notebook logic)
    emo_cols = [c for c in df.columns if str(c).lower().startswith("emotion")]
    emo_cols = sorted(emo_cols)[:3]  # Emotion1, Emotion2, Emotion3 (or variants)
    if len(emo_cols) < 3:
        emo_cols = ["Emotion1", "Emotion2", "Emotion3"]  # fallback

    for _, row in df.iterrows():
        raw = [clean_tok(row.get(c, "")) for c in emo_cols]

        # normalize placeholders
        raw = [("" if r in PLACEHOLDERS else r) for r in raw]

        # handle the "Emotion1" placeholder behavior
        if raw[1] == "emotion1": raw[1] = raw[0]
        if raw[2] == "emotion1": raw[2] = raw[0]

        for r in raw:
            if not r or r in PLACEHOLDERS:
                continue
            if r not in valid:
                ctr[r] += 1

    return ctr

def extract_other_and_notes(df: pd.DataFrame) -> Tuple[Counter, pd.DataFrame]:
    """
    Returns:
      - Counter of 'other' labels (tokenized)
      - DataFrame of rows where other/note exists (for qualitative inspection)
    """
    other_col, note_col = detect_other_note_cols(df)

    other_ctr = Counter()
    rows = []

    for i, row in df.iterrows():
        other_raw = row.get(other_col, "") if other_col else ""
        note_raw  = row.get(note_col, "")  if note_col  else ""

        other_tokens = split_other_field(other_raw) if other_col else []
        for t in other_tokens:
            other_ctr[t] += 1

        other_clean = clean_tok(other_raw)
        note_clean  = str(note_raw).strip() if pd.notna(note_raw) else ""

        if (other_clean and other_clean not in PLACEHOLDERS) or (note_clean and note_clean.lower() not in PLACEHOLDERS):
            rows.append({
                "item_index": i,
                "other_raw": str(other_raw) if pd.notna(other_raw) else "",
                "note_raw": str(note_raw) if pd.notna(note_raw) else "",
            })

    qual_df = pd.DataFrame(rows)
    return other_ctr, qual_df

# --- main extraction per annotator ---

def summarize_out_of_taxonomy(ann_dfs: Dict[str, pd.DataFrame], emotions: List[str], top_n: int = 30):
    summaries = {}
    qual_tables = {}

    for ann_id, df in ann_dfs.items():
        non_plutchik_ctr = extract_non_plutchik_from_emotion_cols(df, emotions)
        other_ctr, qual_df = extract_other_and_notes(df)

        summaries[ann_id] = {
            "non_plutchik_in_emotion_cols": non_plutchik_ctr,
            "other_field_labels": other_ctr,
        }
        qual_tables[ann_id] = qual_df

        print(f"\n=== {ann_id} ===")
        if len(non_plutchik_ctr):
            print(f"Non-Plutchik tokens in Emotion1/2/3 (top {top_n}):")
            print(pd.DataFrame(non_plutchik_ctr.most_common(top_n), columns=["label", "count"]))
        else:
            print("No non-Plutchik tokens found in Emotion1/2/3.")

        if len(other_ctr):
            print(f"\nParsed labels from 'Other' field (top {top_n}):")
            print(pd.DataFrame(other_ctr.most_common(top_n), columns=["label", "count"]))
        else:
            print("\nNo 'Other' labels found (or 'Other' column missing).")

        print(f"\nRows with Other/Note text: {len(qual_df)}")
        # display first few for sanity
        if len(qual_df):
            print(qual_df.head(30))

    return summaries, qual_tables

# --- run it ---
summaries, qual_tables = summarize_out_of_taxonomy(ann_dfs, EMOTIONS, top_n=30)

# save per-annotator qualitative tables
for ann_id, qdf in qual_tables.items():
    qdf.to_csv(f"other_notes_{ann_id}.csv", index=False)